# ML Feature Engineering — Customer-Level Churn Features

Builds a per-customer feature table on top of `fact_sales_enriched` for downstream churn modeling. Output is persisted as a Delta table named `ml_customer_features` with one row per customer and a synthetic `churn_label`.

**Why `fact_sales_enriched` (not `fact_sales`):** the enriched table already carries the broadcast-joined dim attributes (`loyalty_tier`, `region`, `country`, `category`, `brand`, `promotion_type`, `store_type`) AND the pre-computed measures (`gross_revenue`, `discount_amount`, `net_revenue`, `cost`, `gross_margin`). Reading from it lets feature engineering be a pure aggregation — no joins, no re-derivation of measures. That's worth a lot at 100M rows: each join we'd otherwise add costs another shuffle (or a broadcast that pressures executor memory), and re-computing `gross_revenue = quantity × unit_price` on the fly burns CPU on every cell rerun.

**Pipeline:**
1. Load `fact_sales_enriched` via `spark.table()`.
2. Pin a reproducible snapshot date from the data itself (not wall-clock).
3. Aggregate features in a single `groupBy(customer_id)` pass, leveraging both pre-computed measures and pre-joined categorical attributes.
4. Derive ratio/recency features and the synthetic `churn_label` post-aggregation.
5. Persist as Delta (`mode("overwrite")`) and display.

**Design choices that matter at scale:**

- **Single `groupBy` pass.** Every additional pass over a 100M-row fact costs ~5GB of I/O and a full shuffle on `customer_id`. We compute all features in one aggregation — one scan, one shuffle, many outputs.
- **Use the enrichment, don't redo it.** `gross_revenue`, `discount_amount`, `cost`, and `gross_margin` are already materialized in `fact_sales_enriched`. We `sum()` them directly instead of re-deriving from `quantity * unit_price * (1 - discount_pct)` etc. — saves the projection node and keeps the aggregator focused on summation.
- **`F.first(...)` for static customer attributes.** `loyalty_tier` is fixed per `customer_id` (it lives on `dim_customer`), so `F.first("loyalty_tier")` inside the groupBy pulls it through the aggregation without needing a second join. With `ignorenulls=True` it's robust to any NULLs that sneak in.
- **`approx_count_distinct` over `countDistinct`.** Exact distinct counts require per-group sets that grow with cardinality and can OOM on hot customers. HLL sketches use a fixed ~2KB per group and are accurate to ±5% by default — fine for ML features.
- **Snapshot date from the data, not `current_date()`.** ML features must be reproducible: re-running the pipeline tomorrow on the same input must produce the same features. Wall-clock dates break that contract. We use `max(transaction_date)` over the fact as the reference point.
- **No `collect()` on row data.** The only driver materialization is a single scalar (`max(transaction_date)`) via `.first()` — bounded regardless of fact size.
- **Delta `mode("overwrite")` with `overwriteSchema=true`.** Pipeline is re-runnable; schema can evolve as features are added/removed without manual table-drop steps.

In [0]:
from pyspark.sql import functions as F

# Standalone notebook: assume `spark` is available (Databricks default) but import
# explicitly for OSS-Spark/local-dev runs.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

## 1. Load `fact_sales_enriched`

`spark.table()` returns a lazy DataFrame backed by the Delta table — no data is read until an action runs. The columnar Parquet scan will only materialize the columns referenced in the `select(...)` below.

In [0]:
enriched = spark.table("fact_sales_enriched")

# Quick sanity check — schema only, not row data. printSchema() is metadata-only and does
# not trigger a scan.
enriched.printSchema()

## 2. Pin the snapshot date

Reference date for the recency feature. Using `max(transaction_date)` over the fact (rather than `current_date()`) makes the pipeline reproducible: same input → same features, regardless of when it runs.

`.first()` materializes one Row to the driver — bounded cost regardless of fact size, and the only driver materialization in this notebook.

In [0]:
snapshot_date = enriched.agg(F.max("transaction_date").alias("d")).first()["d"]
print(f"Feature snapshot date: {snapshot_date}")

## 3. Project to ONLY the columns the aggregation reads

Even on a Delta-backed source, an explicit `select()` makes the columnar prune visible in the plan and prevents accidental "carry along everything" if the source schema grows.

We pull TWO categories of columns from the enrichment:
- **Pre-computed measures** (`gross_revenue`, `discount_amount`, `net_revenue`, `cost`, `gross_margin`) — sum them directly, no re-derivation.
- **Pre-joined dim attributes** (`loyalty_tier`, `region`, `category`, `brand`, `promotion_type`, `store_id`) — feed `F.first(...)` for customer-static attributes and `approx_count_distinct(...)` for diversity features. No second join needed.

In [0]:
feature_input = enriched.select(
    "customer_id",
    "transaction_date",
    # Pre-computed measures from enrichment — sum directly, do not re-derive.
    "gross_revenue",
    "discount_amount",
    "net_revenue",
    "cost",
    "gross_margin",
    "quantity",
    "discount_pct",
    # Pre-joined dim attributes — fed to F.first() (customer-static) and
    # approx_count_distinct() (diversity).
    "loyalty_tier",
    "region",
    "category",
    "brand",
    "promotion_type",
    "store_id",
)

## 4. Compute all features in a single `groupBy` pass

One scan of the 100M-row fact, one shuffle on `customer_id`, all aggregations computed in the same partial+final HashAggregate pair. Adding a feature later costs an extra aggregator slot, not an extra pass over the data.

Three feature families:
- **Spend volume** — direct sums of the pre-computed measures (`gross_revenue`, `discount_amount`, `net_revenue`, `cost`, `gross_margin`, `quantity`).
- **Behavior / diversity** — promotion usage, category/store/brand diversity. Includes a `promoted_transaction_count` derived via `sum(when(promotion_type IS NOT NULL))` — counts only rows that actually used a promotion, so the ratio `promoted_transaction_count / transaction_count` becomes a clean "promo affinity" feature.
- **Customer-static attributes** — `loyalty_tier`, `region` carried through via `F.first(..., ignorenulls=True)`. These are 1:1 with `customer_id` (they live on `dim_customer` / `dim_store`), so taking the first observed value per group is correct and free.
- **Temporal** — `min`/`max(transaction_date)` for tenure and recency.

Derived features (`avg_transaction_value`, `gross_margin_pct`, `promo_transaction_pct`, `days_since_last_purchase`, `customer_tenure_days`, `churn_label`) are computed AFTER the aggregation, on the small per-customer result — no extra fact scan.

In [0]:
customer_features = (
    feature_input
    .groupBy("customer_id")
    .agg(
        # --- Spend volume — direct sums of pre-computed measures. ----------------
        # gross_revenue / discount_amount / net_revenue / cost / gross_margin all
        # come from the enrichment pipeline; we never re-derive them here.
        F.round(F.sum("gross_revenue"), 2).alias("total_gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("net_revenue"), 2).alias("total_revenue"),
        F.round(F.sum("cost"), 2).alias("total_cost"),
        F.round(F.sum("gross_margin"), 2).alias("total_gross_margin"),
        F.sum("quantity").cast("long").alias("total_quantity"),
        F.count("*").alias("transaction_count"),

        # --- Behavior — promo affinity and avg discount. -------------------------
        # promoted_transaction_count uses sum(when(...)) instead of countDistinct,
        # which is a single integer accumulator per group (cheap) vs an HLL sketch.
        F.sum(F.when(F.col("promotion_type").isNotNull(), 1).otherwise(0))
          .alias("promoted_transaction_count"),
        # coalesce(discount_pct, 0) treats unpromoted transactions as 0% discount so
        # customers who rarely use promotions get a low avg, not NULL.
        F.round(F.avg(F.coalesce("discount_pct", F.lit(0.0))), 4).alias("avg_discount_pct"),

        # --- Diversity — approx_count_distinct uses HyperLogLog with bounded -----
        # ~2KB/group state. Default rsd=0.05 (5% error). Safe default to avoid OOM
        # on hot customers if cardinality grows.
        F.approx_count_distinct("category").alias("distinct_categories"),
        F.approx_count_distinct("brand").alias("distinct_brands"),
        F.approx_count_distinct("store_id").alias("distinct_stores"),
        F.approx_count_distinct("promotion_type").alias("distinct_promotion_types"),

        # --- Customer-static attributes — 1:1 with customer_id, so first() is ----
        # both correct and free. ignorenulls=True is defensive: if a customer's
        # earliest transaction happens to lack the dim attribute (shouldn't happen
        # given LEFT joins in enrichment), we fall through to a non-null observation.
        F.first("loyalty_tier", ignorenulls=True).alias("loyalty_tier"),
        F.first("region", ignorenulls=True).alias("primary_region"),

        # --- Temporal ------------------------------------------------------------
        F.min("transaction_date").alias("first_purchase_date"),
        F.max("transaction_date").alias("last_purchase_date"),
    )

    # ============= Derived features (post-agg, no extra fact scan) =================

    # avg_transaction_value = total_revenue / transaction_count.
    .withColumn(
        "avg_transaction_value",
        F.round(F.col("total_revenue") / F.col("transaction_count"), 2),
    )
    # gross_margin_pct = total_gross_margin / total_revenue. Captures customer
    # profitability — a high-revenue customer who only buys deeply discounted items
    # has a different churn profile than one buying full-price.
    .withColumn(
        "gross_margin_pct",
        F.round(F.col("total_gross_margin") / F.col("total_revenue"), 4),
    )
    # promo_transaction_pct = fraction of transactions that used a promotion.
    .withColumn(
        "promo_transaction_pct",
        F.round(F.col("promoted_transaction_count") / F.col("transaction_count"), 4),
    )
    # customer_tenure_days = how long they've been a customer (first purchase to
    # snapshot). Helps the model distinguish "new customer who hasn't bought in 60
    # days" (might churn) from "long-tenured customer with a 60-day gap" (less worrying).
    .withColumn(
        "customer_tenure_days",
        F.datediff(F.lit(snapshot_date), F.col("first_purchase_date")),
    )
    # days_since_last_purchase: anchored to the snapshot_date scalar (broadcast as
    # a literal). Positive for any customer whose last purchase predates the snapshot.
    .withColumn(
        "days_since_last_purchase",
        F.datediff(F.lit(snapshot_date), F.col("last_purchase_date")),
    )
    # Synthetic churn label. > 60 days since last purchase = churned. Cast to int
    # because downstream ML libraries (MLlib, sklearn-via-pandas-on-Spark) expect
    # integer binary labels rather than booleans.
    .withColumn("churn_label", (F.col("days_since_last_purchase") > F.lit(60)).cast("int"))
)

## 5. Persist as Delta

`mode("overwrite")` for re-runnability. `overwriteSchema=true` so this cell tolerates adding/removing features in future iterations without manual table-drop steps.

No partitioning: the table is one row per customer (~200M rows max), and downstream ML code typically wants a full scan rather than partition pruning. If a downstream join back to `dim_customer` is on the critical path, mirror its `bucketBy(32, "customer_id")` layout here.

In [0]:
(
    customer_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_customer_features")
)

## 6. Inspect the result

Read back via `spark.table()` rather than reusing the `customer_features` DataFrame so the displays reflect what was actually persisted (catches schema-evolution surprises). Three checks:

1. **Schema** — confirms all aggregated and derived columns landed.
2. **Label balance** — useful sanity signal for the synthetic churn definition. A 100% / 0% split usually means the synthetic data has no temporal spread, or the threshold needs tuning.
3. **Sample rows** — quick visual scan for plausibility.

A useful follow-up for actual modeling would be a per-`loyalty_tier` churn rate breakdown — that's both a feature-quality signal (does loyalty correlate with retention?) and a class-balance check by segment.

In [0]:
features_table = spark.table("ml_customer_features")

print("ml_customer_features schema:")
features_table.printSchema()

print("\nLabel distribution (overall):")
display(
    features_table
    .groupBy("churn_label")
    .agg(F.count("*").alias("customers"))
    .orderBy("churn_label")
)

print("\nChurn rate by loyalty_tier (feature-quality sanity check):")
display(
    features_table
    .groupBy("loyalty_tier")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("churn_label"), 4).alias("churn_rate"),
        F.round(F.avg("total_revenue"), 2).alias("avg_total_revenue"),
    )
    .orderBy("loyalty_tier")
)

print("\nSample feature rows:")
display(features_table)